# Lambda Functions

Goal: browse the different ways `lambda` is used in Python — basic anonymous functions, default arguments, sorting/filtering keys, `map`/`filter`/`reduce`, closures returned from functions, dispatch tables, a common loop-capture pitfall, and composing/immediately-invoking lambdas.

## 1. Basic lambda vs. `def`

A `lambda` is a small anonymous function limited to a single expression — the expression's value is returned automatically, with no `return` keyword.

In [ ]:
square = lambda n: n * n
add = lambda a, b: a + b

print(square(5))
print(add(2, 3))

def square_def(n):
    return n * n

print(square(5) == square_def(5))

25
5
True


## 2. Default and keyword arguments

Lambdas support the same argument features as regular functions: default values and keyword arguments.

In [ ]:
greet = lambda name, greeting="Hello": f"{greeting}, {name}!"

print(greet("Alice"))
print(greet("Bob", "Hi"))
print(greet(name="Carol", greeting="Hey"))

Hello, Alice!
Hi, Bob!
Hey, Carol!


## 3. `sorted()` with a lambda `key`

The most common use of `lambda` — a throwaway function that tells `sorted()` (or `min`/`max`) what to sort by, without needing a separate named function.

In [ ]:
numbers = [5, 2, 8, 1, 9]

print(sorted(numbers))
print(sorted(numbers, key=lambda n: -n))  # descending, without writing a named function

words = ["banana", "kiwi", "fig", "apple"]
print(sorted(words, key=lambda w: len(w)))  # sort by length instead of alphabetically

[1, 2, 5, 8, 9]
[9, 8, 5, 2, 1]
['fig', 'kiwi', 'apple', 'banana']


## 4. Sorting a list of dicts

`lambda` shines when the sort key is a small expression on structured data — pulling a field out of each dict.

In [ ]:
people = [
    {"name": "Alice", "age": 30},
    {"name": "Bob", "age": 25},
    {"name": "Carol", "age": 35},
]

by_age = sorted(people, key=lambda p: p["age"])
print([p["name"] for p in by_age])

by_name_len = sorted(people, key=lambda p: len(p["name"]), reverse=True)
print([p["name"] for p in by_name_len])

['Bob', 'Alice', 'Carol']
['Alice', 'Carol', 'Bob']


## 5. `map()`, `filter()`, `reduce()`

Lambdas are a natural fit for the one-off transform/predicate/combine functions these higher-order functions expect.

In [ ]:
numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

doubled = list(map(lambda n: n * 2, numbers))
print(doubled)

evens = list(filter(lambda n: n % 2 == 0, numbers))
print(evens)

from functools import reduce
total = reduce(lambda acc, n: acc + n, numbers, 0)
print(total)

[2, 4, 6, 8, 10, 12, 14, 16, 18, 20]
[2, 4, 6, 8, 10]
55


## 6. Returning a lambda (closures)

A `lambda` returned from a function is a closure — it keeps access to the enclosing function's variables even after that function has returned.

In [ ]:
def make_multiplier(factor):
    return lambda n: n * factor  # closure: remembers `factor` after make_multiplier returns

double = make_multiplier(2)
triple = make_multiplier(3)

print(double(5))
print(triple(5))
print([double(n) for n in range(1, 5)])

10
15
[2, 4, 6, 8]


## 7. Lambdas as dispatch table values

Storing lambdas in a dict is a compact alternative to an `if`/`elif` chain when picking behavior by name.

In [ ]:
operations = {
    "add": lambda a, b: a + b,
    "sub": lambda a, b: a - b,
    "mul": lambda a, b: a * b,
    "div": lambda a, b: a / b,
}

for name, op in operations.items():
    print(name, op(10, 4))

print(operations["mul"](6, 7))

add 14
sub 6
mul 40
div 2.5
42


## 8. Pitfall: late binding in a loop

A `lambda` created inside a loop captures the *variable*, not its value at the time of creation — by the time the lambdas run, the loop variable holds its final value. Giving the loop variable as a default argument captures the value at definition time instead.

In [ ]:
# Pitfall: lambdas inside a loop capture the *variable*, not its value at creation time
funcs_wrong = [lambda: i for i in range(3)]
print([f() for f in funcs_wrong])  # all 2, not 0 1 2

# fix: bind the current value as a default argument
funcs_fixed = [lambda i=i: i for i in range(3)]
print([f() for f in funcs_fixed])  # 0 1 2

[2, 2, 2]
[0, 1, 2]


## 9. Composing and immediately invoking lambdas

Lambdas can call other lambdas to build up new behavior, and — like any function — can be invoked immediately after being defined.

In [ ]:
is_even = lambda n: n % 2 == 0
is_positive = lambda n: n > 0

# combining lambdas: build a new predicate out of existing ones
both = lambda n: is_even(n) and is_positive(n)

numbers = [-4, -3, -2, -1, 0, 1, 2, 3, 4]
print(list(filter(both, numbers)))

# an immediately-invoked lambda
print((lambda a, b: a * b)(6, 7))

[2, 4]
42
